# RoadAnomaly — Notebook final (parcours académique)

**Objectif :** permettre au correcteur de suivre le projet sans lire tout le dépôt.

- Protocole figé : split 42/9/9, entrée 512×288.
- Expériences **exécutées** : E0, E1, E2a, E2b (val + test one-shot).
- Test : Stage 5 terminé — voir phase4_final_test_comparison.csv.
- Aucun token dans ce notebook.


## 1. Identité du projet

Segmentation sémantique binaire supervisée d’anomalies routières.
Référence motivationnelle : Lis et al., ICCV 2019 (méthode différente : resynthèse).

Voir `docs/RAPPORT_TECHNIQUE_CRISP_DM.md`.

In [ ]:
from pathlib import Path
import json, pandas as pd
ROOT = Path("..").resolve() if Path("../experiments").exists() else Path(".").resolve()
print("ROOT", ROOT)
print("split_lock", (ROOT/"experiments/metadata/split_lock.json").exists())
cmp = ROOT/"experiments/results/phase4/phase4_E0_E1_E2_val_comparison.csv"
print(pd.read_csv(cmp).to_string(index=False) if cmp.exists() else "comparison CSV missing")

## 2. Business & Data Understanding

Sources : `reports/01_business_and_data_understanding.md`, figures `reports/figures/01`–`11`.
Faits clés : 60 images, ~9,85 % pixels anomalie, pas de normal-only.

## 3. Data Preparation

Code : `src/preprocessing.py`, `dataset.py`, `augmentation.py`.
Doc : `docs/08_phase2_data_pipeline.md`.
Split hash attendu : `9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78`.

In [ ]:
import sys
sys.path.insert(0, str(ROOT/"src"))
from dataset import load_split_manifest, assignment_sha256_from_manifest, verify_split_lock
manifest = load_split_manifest()
print(manifest["split"].value_counts().to_dict())
print("sha", verify_split_lock(manifest))

## 4. Modeling

| Exp | Code / config |
| --- | --- |
| E0 | `src/models/baseline.py` |
| E1 | `src/models/cnn.py`, metadata `E1_cnn_scratch_20261108_1510.json` |
| E2a/E2b | `src/models/unet_mobilenetv2.py`, `configs/phase4.json` |

Entraînement E2 : notebook `02_phase4_training_colab.ipynb` (déjà exécuté).

## 5. Evaluation (validation)

Voir `docs/13_phase4_results_and_analysis.md`.
Meilleur modèle val : **E2b**, seuil **0,55**, Dice ≈ **0,778**.

In [ ]:
e2b = json.loads((ROOT/"experiments/metadata/phase4/E2b_unet_mobilenetv2_finetune_20261108_1721.json").read_text(encoding="utf-8"))
print("E2b thr", e2b["threshold_selection"]["value"])
print("E2b val mean dice", e2b["validation_metrics"]["mean_per_image"]["mean_dice"])
print("status", e2b["status"])
freeze = json.loads((ROOT/"experiments/metadata/phase4/FINAL_MODEL_SELECTION_FREEZE.json").read_text(encoding="utf-8"))
print("primary", freeze["primary_model"]["experiment_id"])

## 6. Test final

Stage 5 **terminé** (seuils figés, pas de retuning).
Primary **E2b** : Dice test mean/image ≈ **0,818** (global ≈ **0,871**).
Détail : docs/14_final_test_evaluation.md.


In [ ]:
test_csv = ROOT/"experiments/results/phase4/phase4_final_test_comparison.csv"
if test_csv.exists():
    print(pd.read_csv(test_csv).to_string(index=False))
else:
    print("Test results not present locally yet — run Stage 5 Colab notebook; do not invent metrics.")

## 7. Limites & conclusion

Petit jeu ; déséquilibre ; pas de lost-tire en test ; AE non justifié ; démo Streamlit non livrée (proposition seulement).
Rapport : `docs/RAPPORT_TECHNIQUE_CRISP_DM.md`.